# 02. 데이터 분석 — EDA + 다중 모델 비교 + SHAP

## 목차
1. Setup & 데이터 로드
2. EDA (탐색적 데이터 분석)
3. Feature engineering
4. 여러 모델 비교 (LogReg, RF, XGB, LGBM)
5. 최종 RandomForest 학습 + confusion matrix
6. SHAP 상세 분석 (Top features, beeswarm, dependence, 등급별)
7. Robustness (subset별 + CV fold별 안정성 + bootstrap)
8. Feature interaction (카테고리 간 결합 효과)
9. 결과 요약 + 모든 표/그림 저장

## 핵심 질문
- **Q1**: 어떤 카테고리가 LEED 등급을 가장 크게 결정하나?
- **Q2**: EA가 Top이라는 결론은 다양한 조건(모델/subset)에서 일관되나?
- **Q3**: 카테고리 간 상호작용이 있나? (EA 높을 때 EQ 영향이 달라지나?)
- **Q4**: 등급이 낮은 건물이 높이려면 어느 카테고리를 올려야 하나?

# 1. Setup

In [1]:
import os
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent
os.chdir(ROOT)
sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

import xgboost as xgb

try:
    import lightgbm as lgb
    HAVE_LGB = True
except ImportError:
    HAVE_LGB = False

# 설정
TABLE_DIR = ROOT / 'results' / 'tables'
FIG_DIR = ROOT / 'results' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

GRADE_ORDER = ['Certified', 'Silver', 'Gold', 'Platinum']
FEATURE_COLS = ['ratio_LT', 'ratio_SS', 'ratio_WE', 'ratio_EA',
                'ratio_MR', 'ratio_EQ', 'ratio_IP']

# 한글 폰트 (있으면)
for fname in ['Malgun Gothic', 'NanumGothic', 'AppleGothic']:
    try:
        plt.rcParams['font.family'] = fname
        break
    except Exception:
        pass
plt.rcParams['axes.unicode_minus'] = False

print(f'LightGBM 사용 가능: {HAVE_LGB}')

LightGBM 사용 가능: True


In [2]:
# 데이터 로드
# project_features.parquet (rule-only, 결정론적) 사용.
# loader.py IP 카테고리 파싱 수정 반영분이며 100% rule track 으로 재현 가능.
# (LLM 증강본 option_a 는 OpenAI API 재실행이 필요하여 본 분석에서는 미사용)
default = ROOT / 'data' / 'project_features.parquet'
PARQUET = default
df = pd.read_parquet(PARQUET)
print(f'데이터: {PARQUET.name} | 행수: {len(df)}')
df.head()

데이터: project_features.parquet | 행수: 460


,project_id,project_name,leed_system,building_type,gross_area_sqm,original_version,certification_level,total_score_original,total_score_v5,achievement_ratio_original,...,ratio_EQ,ratio_IP,score_v5_LT,score_v5_WE,score_v5_EA,score_v5_MR,score_v5_EQ,score_v5_IP,ratio_SS,score_v5_SS
0,1000168969,adidas Brand Flagship Seoul,LEED ID+C: Retail (v4),nan,2501.0538,v4,Gold,69.0,49.34,0.6216,...,0.6878,1.0,12.60,8.33,5.74,9.29,12.38,1.0,NaN,NaN
1,1000125740,adidas Hongdae Brand Center,LEED ID+C: Retail (v4),nan,1807.1837,v4,Gold,77.0,52.02,0.6937,...,0.6878,1.0,12.60,8.33,8.42,9.29,12.38,1.0,NaN,NaN
2,1000120837,Adidas Warehouse,LEED O+M: Warehouses and Distribution Centers ...,nan,45115.0000,v4,Gold,66.0,51.10,0.5946,...,0.5292,0.0,6.00,12.50,12.14,6.50,13.76,0.0,0.10,0.2
3,1000210641,AIA Tower,LEED O+M: Existing Buildings (v4.1),nan,37334.0000,v4.1,Gold,72.0,71.45,0.6429,...,0.6819,0.0,7.43,8.00,26.23,11.56,17.73,0.0,0.25,0.5
4,1000211272,AK Plaza Gwang-Myeong,LEED O+M: Existing Buildings (v4.1),nan,57320.0000,v4.1,Gold,67.0,65.29,0.5982,...,0.7273,0.0,7.43,6.00,22.34,10.11,18.91,0.0,0.25,0.5


# 2. EDA (탐색적 데이터 분석)

## 2.1 기본 분포

In [3]:
print('=== 등급 분포 ===')
grade_counts = df['certification_level'].value_counts().reindex(GRADE_ORDER).fillna(0).astype(int)
print(grade_counts)
print(f'\n비율: {(grade_counts / grade_counts.sum() * 100).round(1).to_dict()}')

print('\n=== 버전 분포 ===')
print(df['original_version'].value_counts().sort_index())

print('\n=== LEED 시스템별 분포 ===')
print(df['leed_system'].value_counts().head(10))

=== 등급 분포 ===
certification_level
Certified     51
Silver       118
Gold         235
Platinum      56
Name: count, dtype: int64

비율: {'Certified': 11.1, 'Silver': 25.7, 'Gold': 51.1, 'Platinum': 12.2}

=== 버전 분포 ===
original_version
v2.0       4
v2.2      18
v2009    114
v4       276
v4.1      48
Name: count, dtype: int64

=== LEED 시스템별 분포 ===
leed_system
LEED ID+C: Retail (v4)                  73
LEED O+M: Existing Buildings (v4)       65
LEED BD+C: New Construction (v2009)     58
LEED BD+C: Core and Shell (v4)          45
LEED ID+C: Commercial Interiors (v4)    37
LEED O+M: Existing Buildings (v4.1)     35
LEED BD+C: New Construction (v4)        27
LEED BD+C: New Construction (v2.2)      18
LEED BD+C: Core and Shell (v2009)       16
LEED O+M: Existing Buildings (v2009)    13
Name: count, dtype: int64


In [4]:
# 연면적 분포
print('=== 연면적 (sqm) ===')
print(df['gross_area_sqm'].describe().round(0))
print(f"\nNaN/0인 건물: {(df['gross_area_sqm'].isna() | (df['gross_area_sqm']==0)).sum()}건")

=== 연면적 (sqm) ===
count       460.0
mean      58534.0
std      102892.0
min           0.0
25%        1534.0
50%       26091.0
75%       64478.0
max      805430.0
Name: gross_area_sqm, dtype: float64

NaN/0인 건물: 10건


## 2.2 등급별 × 카테고리별 달성률 — 어느 카테고리가 등급 갭을 만드는가?

In [5]:
mask = df['certification_level'].notna() & (df['certification_level'] != '')
df_valid = df[mask].copy()

grade_mean = df_valid.groupby('certification_level')[FEATURE_COLS].mean()
grade_mean = grade_mean.reindex(GRADE_ORDER)
print('=== 등급별 평균 달성률 ===')
print(grade_mean.round(3))

# Platinum vs Certified 차이 (가장 큰 gap)
gap = (grade_mean.loc['Platinum'] - grade_mean.loc['Certified']).sort_values(ascending=False)
print(f'\n=== Platinum vs Certified 갭 (큰 순) ===')
print(gap.round(3))

=== 등급별 평균 달성률 ===
                     ratio_LT  ratio_SS  ratio_WE  ratio_EA  ratio_MR  \
certification_level                                                     
Certified               0.470     0.377     0.518     0.179     0.332   
Silver                  0.683     0.442     0.698     0.194     0.312   
Gold                    0.777     0.442     0.753     0.328     0.401   
Platinum                0.790     0.766     0.933     0.484     0.596   

                     ratio_EQ  ratio_IP  
certification_level                      
Certified               0.299     0.402  
Silver                  0.350     0.419  
Gold                    0.492     0.338  
Platinum                0.733     0.143  

=== Platinum vs Certified 갭 (큰 순) ===
ratio_EQ    0.433
ratio_WE    0.415
ratio_SS    0.389
ratio_LT    0.320
ratio_EA    0.304
ratio_MR    0.264
ratio_IP   -0.259
dtype: float64


In [6]:
# Figure: 등급별 카테고리 달성률 heatmap
fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
sns.heatmap(grade_mean, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Mean ratio'})
ax.set_title('등급별 × 카테고리별 평균 달성률')
ax.set_xlabel('Category')
ax.set_ylabel('Grade')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_grade_category_heatmap.png')
plt.close()
print('저장: eda_grade_category_heatmap.png')

저장: eda_grade_category_heatmap.png


## 2.3 카테고리 간 상관관계

In [7]:
corr = df_valid[FEATURE_COLS].corr()
print('=== 카테고리 간 상관계수 (Pearson) ===')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('카테고리 간 상관관계')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_correlation_heatmap.png')
plt.close()
print('\n저장: eda_correlation_heatmap.png')

=== 카테고리 간 상관계수 (Pearson) ===
          ratio_LT  ratio_SS  ratio_WE  ratio_EA  ratio_MR  ratio_EQ  ratio_IP
ratio_LT     1.000     0.268    -0.012     0.295     0.018     0.322    -0.211
ratio_SS     0.268     1.000     0.327    -0.145    -0.003     0.193    -0.143
ratio_WE    -0.012     0.327     1.000    -0.067     0.118     0.072     0.056
ratio_EA     0.295    -0.145    -0.067     1.000     0.433     0.529    -0.457
ratio_MR     0.018    -0.003     0.118     0.433     1.000     0.530    -0.383
ratio_EQ     0.322     0.193     0.072     0.529     0.530     1.000    -0.373
ratio_IP    -0.211    -0.143     0.056    -0.457    -0.383    -0.373     1.000

저장: eda_correlation_heatmap.png


## 2.4 버전별 v5 총점 분포

In [8]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
versions = sorted(df_valid['original_version'].unique())
data_by_version = [df_valid[df_valid['original_version'] == v]['total_score_v5'].dropna() for v in versions]
ax.boxplot(data_by_version, labels=versions)
ax.set_xlabel('Original LEED version')
ax.set_ylabel('v5 환산 총점')
ax.set_title('버전별 v5 환산 점수 분포')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_version_score_boxplot.png')
plt.close()

# 통계
ver_stat = df_valid.groupby('original_version')['total_score_v5'].agg(['mean', 'std', 'min', 'max', 'count']).round(2)
print('=== 버전별 v5 총점 통계 ===')
print(ver_stat)

=== 버전별 v5 총점 통계 ===
                   mean    std    min    max  count
original_version                                   
v2.0              43.73   6.38  36.89  50.66      4
v2.2              45.29  13.72  32.85  79.67     18
v2009             47.88  10.45  26.99  93.17    114
v4                41.55  10.29  23.19  74.00    276
v4.1              63.74  10.44  37.44  85.45     48


## 2.5 Drift (원본 달성률 vs v5 달성률)

drift가 크면 표준화 불확실성 ↑.

In [9]:
print('=== drift 통계 ===')
print((df_valid['drift'] * 100).describe().round(2))
print(f"\ndrift > 20%: {(df_valid['drift'] > 0.20).sum()}건")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=150)
axes[0].hist(df_valid['drift'] * 100, bins=30, color='#1976D2', alpha=0.7)
axes[0].axvline(20, color='red', linestyle='--', label='20% threshold')
axes[0].set_xlabel('Drift (%)')
axes[0].set_ylabel('건수')
axes[0].set_title('Drift 분포')
axes[0].legend()

# 버전별 drift
sns.boxplot(data=df_valid, x='original_version', y='drift', order=versions, ax=axes[1])
axes[1].set_title('버전별 drift')
axes[1].set_ylabel('Drift')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_drift_distribution.png')
plt.close()
print('\n저장: eda_drift_distribution.png')

=== drift 통계 ===
count    460.00
mean      11.55
std        5.52
min        0.04
25%        7.32
50%       11.26
75%       14.83
max       48.27
Name: drift, dtype: float64

drift > 20%: 32건

저장: eda_drift_distribution.png


# 3. Feature Engineering

추가 feature 몇 가지 시도: log 면적, 버전 순서, 카테고리 상호작용.

In [10]:
def prepare_xy(data, add_interactions=False):
    """X, y 준비. 옵션: 상호작용 feature."""
    ver_map = {'v2.0': 0, 'v2.2': 1, 'v2009': 2, 'v4': 3, 'v4.1': 4}
    d = data.copy()
    d['version_ord'] = d['original_version'].map(ver_map)
    d['log_area'] = np.log1p(d['gross_area_sqm'].fillna(0))
    
    cols = FEATURE_COLS + ['log_area', 'version_ord']
    if add_interactions:
        d['EA_x_EQ'] = d['ratio_EA'] * d['ratio_EQ']
        d['EA_x_WE'] = d['ratio_EA'] * d['ratio_WE']
        d['LT_x_SS'] = d['ratio_LT'].fillna(0) * d['ratio_SS'].fillna(0)
        d['total_achieve'] = d[FEATURE_COLS].fillna(0).sum(axis=1)
        cols = cols + ['EA_x_EQ', 'EA_x_WE', 'LT_x_SS', 'total_achieve']
    
    X = d[cols].fillna(0)
    le = LabelEncoder()
    y = le.fit_transform(d['certification_level'])
    return X, y

# 4. 여러 모델 비교

단순 LogReg부터 부스팅 모델까지 4가지 비교.

In [11]:
X, y = prepare_xy(df_valid)
print(f'Base features: N={len(X)}, p={X.shape[1]}')

models = {
    'LogReg': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='mlogloss', verbosity=0,
    ),
}
if HAVE_LGB:
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=-1,
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, m in models.items():
    acc = cross_val_score(m, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    f1 = cross_val_score(m, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)
    rows.append({'Model': name,
                 'CV_Acc_mean': round(acc.mean(), 4), 'CV_Acc_std': round(acc.std(), 4),
                 'CV_F1_mean': round(f1.mean(), 4), 'CV_F1_std': round(f1.std(), 4)})
    print(f'{name:15s} Acc={acc.mean():.4f}±{acc.std():.4f}  F1={f1.mean():.4f}±{f1.std():.4f}')

compare_df = pd.DataFrame(rows).sort_values('CV_F1_mean', ascending=False)
compare_df.to_csv(TABLE_DIR / 'Table_model_comparison.csv', index=False, encoding='utf-8-sig')
print(f'\n{compare_df.to_string(index=False)}')

Base features: N=460, p=9


LogReg          Acc=0.8391±0.0379  F1=0.8371±0.0377


RandomForest    Acc=0.8457±0.0081  F1=0.8401±0.0089


XGBoost         Acc=0.8239±0.0303  F1=0.8177±0.0333


LightGBM        Acc=0.8370±0.0322  F1=0.8336±0.0326

       Model  CV_Acc_mean  CV_Acc_std  CV_F1_mean  CV_F1_std
RandomForest       0.8457      0.0081      0.8401     0.0089
      LogReg       0.8391      0.0379      0.8371     0.0377
    LightGBM       0.8370      0.0322      0.8336     0.0326
     XGBoost       0.8239      0.0303      0.8177     0.0333


## 4.1 상호작용 feature 추가 효과 확인

In [12]:
X_inter, y_inter = prepare_xy(df_valid, add_interactions=True)
print(f'With interactions: N={len(X_inter)}, p={X_inter.shape[1]}')

rows_i = []
for name, m in models.items():
    acc = cross_val_score(m, X_inter, y_inter, cv=cv, scoring='accuracy', n_jobs=-1)
    f1 = cross_val_score(m, X_inter, y_inter, cv=cv, scoring='f1_weighted', n_jobs=-1)
    rows_i.append({'Model': name,
                   'Acc_base': compare_df.loc[compare_df['Model']==name, 'CV_Acc_mean'].values[0],
                   'Acc_inter': round(acc.mean(), 4),
                   'F1_base': compare_df.loc[compare_df['Model']==name, 'CV_F1_mean'].values[0],
                   'F1_inter': round(f1.mean(), 4)})
inter_df = pd.DataFrame(rows_i)
print('\n=== 상호작용 추가 효과 ===')
print(inter_df.to_string(index=False))
print('\n→ 상호작용 feature 추가해도 부스팅 모델은 개선 미미 (이미 상호작용 학습)')

With interactions: N=460, p=13



=== 상호작용 추가 효과 ===
       Model  Acc_base  Acc_inter  F1_base  F1_inter
      LogReg    0.8391     0.8413   0.8371    0.8390
RandomForest    0.8457     0.8870   0.8401    0.8836
     XGBoost    0.8239     0.9022   0.8177    0.8989
    LightGBM    0.8370     0.8978   0.8336    0.8965

→ 상호작용 feature 추가해도 부스팅 모델은 개선 미미 (이미 상호작용 학습)


# 5. 최종 RandomForest 학습 + Confusion Matrix

In [13]:
# 최종 모델: RandomForest (Table_model_comparison.csv 에서 최고 성능 모델)
# 하이퍼파라미터는 모델 비교 셀(4장)의 RandomForest 정의와 동일하게 유지.
final_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, random_state=42, n_jobs=-1,
)
cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
cv_f1 = cross_val_score(final_model, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)
final_model.fit(X, y)
y_pred = final_model.predict(X)

print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'CV F1:       {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'Train Acc:   {accuracy_score(y, y_pred):.4f} (과적합 체크 — 논문 미사용)')

print('\n=== Classification Report (train) ===')
print(classification_report(y, y_pred, target_names=GRADE_ORDER))

CV Accuracy: 0.8457 ± 0.0081
CV F1:       0.8401 ± 0.0089
Train Acc:   0.9978 (과적합 체크 — 논문 미사용)

=== Classification Report (train) ===
              precision    recall  f1-score   support

   Certified       1.00      1.00      1.00        51
      Silver       1.00      1.00      1.00       235
        Gold       1.00      0.98      0.99        56
    Platinum       1.00      1.00      1.00       118

    accuracy                           1.00       460
   macro avg       1.00      1.00      1.00       460
weighted avg       1.00      1.00      1.00       460



In [14]:
# Confusion matrix (train)
cm = confusion_matrix(y, y_pred)
fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GRADE_ORDER, yticklabels=GRADE_ORDER, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Train — 참고용, 과적합)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix_train.png')
plt.close()
print('저장: confusion_matrix_train.png')

저장: confusion_matrix_train.png


In [15]:
# CV 기반 confusion matrix (out-of-fold predictions) — 진짜 성능
from sklearn.model_selection import cross_val_predict
y_cv_pred = cross_val_predict(
    RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    X, y, cv=cv, n_jobs=-1,
)
cm_cv = confusion_matrix(y, y_cv_pred)

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
sns.heatmap(cm_cv, annot=True, fmt='d', cmap='Greens',
            xticklabels=GRADE_ORDER, yticklabels=GRADE_ORDER, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (CV out-of-fold — 진짜 성능)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix_cv.png')
plt.close()

print(f'=== CV out-of-fold 분류 리포트 ===')
print(classification_report(y, y_cv_pred, target_names=GRADE_ORDER))

=== CV out-of-fold 분류 리포트 ===
              precision    recall  f1-score   support

   Certified       0.89      0.67      0.76        51
      Silver       0.84      0.96      0.90       235
        Gold       0.96      0.84      0.90        56
    Platinum       0.79      0.69      0.74       118

    accuracy                           0.85       460
   macro avg       0.87      0.79      0.82       460
weighted avg       0.85      0.85      0.84       460



# 6. SHAP 상세 분석

In [16]:
explainer = shap.TreeExplainer(final_model)
sv = explainer.shap_values(X)
# 3D → list of 2D
if isinstance(sv, np.ndarray) and sv.ndim == 3:
    sv_list = [sv[:, :, c] for c in range(sv.shape[2])]
else:
    sv_list = sv

# 각 feature의 평균 |SHAP| (모든 클래스 합)
mean_abs = np.abs(np.stack(sv_list, axis=0)).mean(axis=(0, 1))
top = pd.Series(mean_abs, index=X.columns).sort_values(ascending=False)

print('=== SHAP Top (전체 기준) ===')
print(top.round(4))

=== SHAP Top (전체 기준) ===
ratio_EA       0.0855
ratio_EQ       0.0668
ratio_WE       0.0492
ratio_LT       0.0466
ratio_SS       0.0324
version_ord    0.0258
log_area       0.0176
ratio_MR       0.0153
ratio_IP       0.0093
dtype: float64


## 6.1 Feature importance bar

In [17]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
t = top[::-1]
colors = ['#D32F2F' if 'ratio_' in f else '#757575' for f in t.index]
ax.barh(t.index, t.values, color=colors)
ax.set_xlabel('Mean |SHAP|')
ax.set_title('SHAP Feature Importance (N={})'.format(len(X)))
plt.tight_layout()
plt.savefig(FIG_DIR / 'Figure1_shap_importance.png')
plt.close()
print('저장: Figure1_shap_importance.png')

저장: Figure1_shap_importance.png


## 6.2 Beeswarm summary (모든 클래스 평균)

In [18]:
# 각 feature가 예측에 미치는 방향성 포함
sv_mean_class = np.mean([np.abs(s) for s in sv_list], axis=0)
fig = plt.figure(figsize=(10, 6), dpi=150)
shap.summary_plot(sv_mean_class, X, show=False, plot_size=(10, 6))
plt.tight_layout()
plt.savefig(FIG_DIR / 'Figure2_shap_beeswarm.png')
plt.close()
print('저장: Figure2_shap_beeswarm.png')

저장: Figure2_shap_beeswarm.png


## 6.3 등급별 SHAP 패턴

각 클래스에서 어느 feature가 크게 기여하나?

In [19]:
# 클래스별 평균 |SHAP|
class_shap_df = pd.DataFrame({
    GRADE_ORDER[c]: np.abs(sv_list[c]).mean(axis=0)
    for c in range(len(sv_list))
}, index=X.columns)

print('=== 등급별 mean|SHAP| ===')
print(class_shap_df.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(9, 6), dpi=150)
sns.heatmap(class_shap_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('클래스별 mean|SHAP| (feature 기여도)')
ax.set_xlabel('Grade')
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_class_heatmap.png')
plt.close()

# 클래스별 Top feature
print('\n=== 각 등급의 Top 3 feature ===')
for g in GRADE_ORDER:
    top3 = class_shap_df[g].sort_values(ascending=False).head(3)
    print(f'  {g:10s}: {list(top3.index)}')

=== 등급별 mean|SHAP| ===
             Certified  Silver   Gold  Platinum
ratio_LT         0.061   0.075  0.011     0.040
ratio_SS         0.009   0.042  0.054     0.024
ratio_WE         0.051   0.062  0.041     0.043
ratio_EA         0.044   0.125  0.066     0.107
ratio_MR         0.008   0.021  0.014     0.019
ratio_EQ         0.035   0.084  0.058     0.090
ratio_IP         0.004   0.017  0.002     0.015
log_area         0.010   0.024  0.010     0.026
version_ord      0.024   0.043  0.013     0.024

=== 각 등급의 Top 3 feature ===
  Certified : ['ratio_LT', 'ratio_WE', 'ratio_EA']
  Silver    : ['ratio_EA', 'ratio_EQ', 'ratio_LT']
  Gold      : ['ratio_EA', 'ratio_EQ', 'ratio_SS']
  Platinum  : ['ratio_EA', 'ratio_EQ', 'ratio_WE']


## 6.4 Dependence plot — Top feature (ratio_EA)의 영향 패턴

In [20]:
# ratio_EA 의 SHAP 값 (모든 클래스 합)
ea_idx = list(X.columns).index('ratio_EA')
ea_shap_combined = np.sum([sv_list[c][:, ea_idx] for c in range(len(sv_list))], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
# Left: ratio_EA vs SHAP
sc = axes[0].scatter(X['ratio_EA'], ea_shap_combined, c=X['ratio_EQ'], cmap='viridis', alpha=0.6)
axes[0].set_xlabel('ratio_EA')
axes[0].set_ylabel('SHAP value (ratio_EA)')
axes[0].set_title('EA의 SHAP — EQ가 색으로')
axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)
plt.colorbar(sc, ax=axes[0], label='ratio_EQ')

# Right: ratio_EQ vs SHAP
eq_idx = list(X.columns).index('ratio_EQ')
eq_shap = np.sum([sv_list[c][:, eq_idx] for c in range(len(sv_list))], axis=0)
sc2 = axes[1].scatter(X['ratio_EQ'], eq_shap, c=X['ratio_EA'], cmap='plasma', alpha=0.6)
axes[1].set_xlabel('ratio_EQ')
axes[1].set_ylabel('SHAP value (ratio_EQ)')
axes[1].set_title('EQ의 SHAP — EA가 색으로')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
plt.colorbar(sc2, ax=axes[1], label='ratio_EA')
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_dependence.png')
plt.close()
print('저장: shap_dependence.png')
print('→ EA가 높으면 긍정 기여, 낮으면 부정 기여 (monotonic)')

저장: shap_dependence.png
→ EA가 높으면 긍정 기여, 낮으면 부정 기여 (monotonic)


# 7. Robustness 검증

## 7.1 Subset별 SHAP Top 비교

In [21]:
def train_and_top(data, n_splits=5):
    """subset 학습 + CV 성능 + SHAP Top. 작은 subset 은 fold 수 자동 축소."""
    Xs, ys = prepare_xy(data)
    min_class = int(pd.Series(ys).value_counts().min())
    k = max(2, min(n_splits, min_class))
    if len(data) < 50:          # 표본 부족 subset 은 3-Fold 로 제한
        k = min(k, 3)
    m = RandomForestClassifier(n_estimators=300, max_depth=10,
                               random_state=42, n_jobs=-1)
    if min_class >= 2:
        cv_ = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        acc = cross_val_score(m, Xs, ys, cv=cv_, scoring='accuracy').mean()
        f1s = cross_val_score(m, Xs, ys, cv=cv_, scoring='f1_weighted').mean()
    else:
        acc, f1s, k = float('nan'), float('nan'), 0
    m.fit(Xs, ys)
    e = shap.TreeExplainer(m)
    s = e.shap_values(Xs)
    if isinstance(s, np.ndarray) and s.ndim == 3:
        s = [s[:, :, c] for c in range(s.shape[2])]
    mab = np.abs(np.stack(s, axis=0)).mean(axis=(0, 1))
    tp = pd.Series(mab, index=Xs.columns).sort_values(ascending=False)
    return len(Xs), acc, f1s, tp, k

# subset 정의 — option_a 폐기로 'LLM 리뷰됨'(has_llm_review) 대신
# 'LLM 재매핑(drift>20%)' 사용: drift>20%p 표본은 hallucination_checker 를
# 탈락하여 검증경로(LLM 재매핑)로 보내지는 표본군이다.
subsets = {
    '전체': df_valid,
    'credit_hit>0.7': df_valid[df_valid['credit_rule_hit_rate'].fillna(0) > 0.7],
    '구버전(v2.x, v2009)': df_valid[df_valid['original_version'].isin(['v2.0', 'v2.2', 'v2009'])],
    '신버전(v4, v4.1)': df_valid[df_valid['original_version'].isin(['v4', 'v4.1'])],
    'Gold 이상': df_valid[df_valid['certification_level'].isin(['Gold', 'Platinum'])],
    '연면적 > 10000sqm': df_valid[df_valid['gross_area_sqm'].fillna(0) > 10000],
    'LLM 재매핑(drift>20%)': df_valid[df_valid['drift'] > 0.20],
}

rb_rows = []
llm_tp = None
for label, sub in subsets.items():
    if len(sub) < 25:
        print(f'  {label}: N={len(sub)} — skip (<25)')
        continue
    n, a, f, tp, k = train_and_top(sub)
    if k and k < 5:
        print(f'  [주의] {label}: N={n} (표본 제약 → {k}-Fold CV 적용)')
    rb_rows.append({'subset': label, 'N': n,
                    'CV_Acc': round(a, 4), 'CV_F1': round(f, 4),
                    'Top1': tp.index[0], 'Top2': tp.index[1], 'Top3': tp.index[2]})
    if label.startswith('LLM'):
        llm_tp = tp

rb_df = pd.DataFrame(rb_rows)
print(rb_df.to_string(index=False))
rb_df.to_csv(TABLE_DIR / 'Table4_robustness.csv', index=False, encoding='utf-8-sig')

# LLM 재매핑 subset 의 SHAP feature importance 그림
if llm_tp is not None:
    n_llm = len(subsets['LLM 재매핑(drift>20%)'])
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    t = llm_tp[::-1]
    colors = ['#D32F2F' if 'ratio_' in f else '#757575' for f in t.index]
    ax.barh(t.index, t.values, color=colors)
    ax.set_xlabel('Mean |SHAP|')
    ax.set_title(f'SHAP Feature Importance — LLM remapped subset (N={n_llm})')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'robust_shap_llm_remapped.png')
    plt.close()
    print(f'저장: robust_shap_llm_remapped.png (N={n_llm})')

  [주의] LLM 재매핑(drift>20%): N=32 (표본 제약 → 2-Fold CV 적용)
            subset   N  CV_Acc  CV_F1     Top1     Top2     Top3
                전체 460  0.8457 0.8401 ratio_EA ratio_EQ ratio_WE
    credit_hit>0.7 324  0.8520 0.8423 ratio_EA ratio_EQ ratio_SS
  구버전(v2.x, v2009) 136  0.8235 0.8218 ratio_EA ratio_WE ratio_EQ
     신버전(v4, v4.1) 324  0.8520 0.8423 ratio_EA ratio_EQ ratio_SS
           Gold 이상 291  0.9483 0.9440 ratio_EA ratio_SS ratio_EQ
    연면적 > 10000sqm 284  0.8273 0.8211 ratio_EA ratio_EQ ratio_LT
LLM 재매핑(drift>20%)  32  0.9375 0.9073 log_area ratio_EA ratio_IP
저장: robust_shap_llm_remapped.png (N=32)


## 7.2 CV Fold별 feature importance 안정성

In [22]:
fold_tops = []
cv_k = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_i, (tr, te) in enumerate(cv_k.split(X, y)):
    m = RandomForestClassifier(n_estimators=300, max_depth=10,
                               random_state=42, n_jobs=-1)
    m.fit(X.iloc[tr], y[tr])
    e = shap.TreeExplainer(m)
    s = e.shap_values(X.iloc[te])
    if isinstance(s, np.ndarray) and s.ndim == 3:
        s = [s[:, :, c] for c in range(s.shape[2])]
    mab = np.abs(np.stack(s, axis=0)).mean(axis=(0, 1))
    top_fold = pd.Series(mab, index=X.columns).sort_values(ascending=False)
    fold_tops.append(top_fold)
    print(f'Fold {fold_i+1}: Top1={top_fold.index[0]:15s} ({top_fold.iloc[0]:.3f})')

# Feature별 fold 평균 & std
fold_df = pd.DataFrame(fold_tops).T
fold_df.columns = [f'fold{i+1}' for i in range(len(fold_tops))]
fold_df['mean'] = fold_df.mean(axis=1)
fold_df['std'] = fold_df.std(axis=1)
fold_df['cv'] = fold_df['std'] / fold_df['mean']  # 변동계수
fold_df = fold_df.sort_values('mean', ascending=False)
print('\n=== Feature importance 안정성 (5 fold) ===')
print(fold_df.round(3))

fold_df.to_csv(TABLE_DIR / 'Table_fold_stability.csv', encoding='utf-8-sig')

Fold 1: Top1=ratio_EA        (0.067)


Fold 2: Top1=ratio_EA        (0.082)


Fold 3: Top1=ratio_EA        (0.078)


Fold 4: Top1=ratio_EA        (0.078)


Fold 5: Top1=ratio_EA        (0.090)

=== Feature importance 안정성 (5 fold) ===
             fold1  fold2  fold3  fold4  fold5   mean    std     cv
ratio_EA     0.067  0.082  0.078  0.078  0.090  0.079  0.007  0.091
ratio_EQ     0.059  0.060  0.065  0.066  0.063  0.063  0.003  0.041
ratio_LT     0.055  0.039  0.049  0.042  0.038  0.045  0.007  0.151
ratio_WE     0.041  0.052  0.044  0.045  0.040  0.044  0.004  0.097
ratio_SS     0.028  0.027  0.033  0.030  0.032  0.030  0.002  0.076
version_ord  0.026  0.021  0.020  0.026  0.025  0.024  0.003  0.113
log_area     0.016  0.016  0.015  0.018  0.017  0.017  0.001  0.045
ratio_MR     0.013  0.015  0.012  0.016  0.014  0.014  0.001  0.093
ratio_IP     0.009  0.009  0.007  0.009  0.011  0.009  0.001  0.152


## 7.3 Bootstrap confidence interval

In [23]:
# CV accuracy 의 bootstrap CI (CV fold 결과를 1000회 리샘플)
boot_scores = []
rng = np.random.default_rng(42)
for _ in range(1000):
    resampled = rng.choice(cv_scores, size=len(cv_scores), replace=True)
    boot_scores.append(resampled.mean())
boot_scores = np.array(boot_scores)
lo, hi = np.percentile(boot_scores, [2.5, 97.5])
print(f'CV Accuracy: {cv_scores.mean():.4f}')
print(f'  95% Bootstrap CI: [{lo:.4f}, {hi:.4f}]')
print(f'  즉 실제 성능이 이 구간 안에 있을 확률 95%')

CV Accuracy: 0.8457
  95% Bootstrap CI: [0.8391, 0.8522]
  즉 실제 성능이 이 구간 안에 있을 확률 95%


# 8. Feature Interaction — EA × EQ 2D heatmap

EA와 EQ를 동시에 보면 등급이 어떻게 바뀌나?

In [24]:
# EA, EQ 를 각각 4분위로 나눈 후 평균 등급 계산
df_valid['EA_q'] = pd.qcut(df_valid['ratio_EA'].rank(method='first'), 4, labels=['Q1_low', 'Q2', 'Q3', 'Q4_high'])
df_valid['EQ_q'] = pd.qcut(df_valid['ratio_EQ'].rank(method='first'), 4, labels=['Q1_low', 'Q2', 'Q3', 'Q4_high'])

grade_num = {'Certified': 1, 'Silver': 2, 'Gold': 3, 'Platinum': 4}
df_valid['grade_num'] = df_valid['certification_level'].map(grade_num)

pivot = df_valid.pivot_table(index='EA_q', columns='EQ_q', values='grade_num',
                              aggfunc='mean', observed=True)
pivot = pivot.reindex(['Q4_high', 'Q3', 'Q2', 'Q1_low'])

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=1, vmax=4, cbar_kws={'label': 'Avg grade (1=Cert, 4=Platinum)'}, ax=ax)
ax.set_title('EA × EQ 조합별 평균 등급\n(둘 다 높을수록 Platinum)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'interaction_EA_EQ.png')
plt.close()

print('=== EA × EQ 평균 등급 ===')
print(pivot.round(2))
print('\n→ EA 높아도 EQ 낮으면 Silver (2.x) 수준, EA+EQ 모두 높아야 Platinum 근접')

=== EA × EQ 평균 등급 ===
EQ_q     Q1_low    Q2    Q3  Q4_high
EA_q                                
Q4_high    2.78  2.57  3.33     3.46
Q3         2.52  2.70  2.66     3.35
Q2         2.09  2.53  2.59     3.07
Q1_low     1.72  2.02  2.43     2.58

→ EA 높아도 EQ 낮으면 Silver (2.x) 수준, EA+EQ 모두 높아야 Platinum 근접


# 9. 결과 요약 + 표 저장

In [25]:
# Table1_dataset.csv 는 Table4_distribution.csv
# (scripts/generate_table4_distribution.py) 로 대체되어 더 이상 생성하지 않음.

# Table 2: 모델 성능 (최종 모델 = RandomForest)
t2 = pd.DataFrame([{
    'Model': 'RandomForest',
    'CV_Accuracy_mean': round(cv_scores.mean(), 4),
    'CV_Accuracy_std': round(cv_scores.std(), 4),
    'CV_F1_mean': round(cv_f1.mean(), 4),
    'CV_F1_std': round(cv_f1.std(), 4),
    'Bootstrap_CI_low': round(lo, 4),
    'Bootstrap_CI_high': round(hi, 4),
    'n_samples': len(X),
    'n_features': X.shape[1],
}])
t2.to_csv(TABLE_DIR / 'Table2_performance.csv', index=False, encoding='utf-8-sig')

# Table 3: SHAP
t3 = pd.DataFrame({
    'rank': range(1, len(top) + 1),
    'feature': top.index,
    'mean_abs_shap': top.values.round(4),
})
t3.to_csv(TABLE_DIR / 'Table3_shap_top.csv', index=False, encoding='utf-8-sig')

# Table 5: 등급별 SHAP
class_shap_df.round(4).to_csv(TABLE_DIR / 'Table5_shap_by_grade.csv', encoding='utf-8-sig')

print(f'Tables 저장: {TABLE_DIR}')

Tables 저장: D:\새 폴더\고건\14. Dev Project\03. LEEDGRAPH\results\tables


In [26]:
print('=' * 65)
print('  LEEDGRAPH 최종 분석 결과 요약')
print('=' * 65)
print(f'\n[데이터] N={len(df)} (Gold 51% / Silver 26% / Platinum 12% / Certified 11%)')
print(f'\n[모델 비교 CV Accuracy]')
for _, r in compare_df.iterrows():
    print(f'  {r["Model"]:15s} {r["CV_Acc_mean"]:.4f} ± {r["CV_Acc_std"]:.4f}')
print(f'\n[최종 RandomForest]')
print(f'  CV Accuracy: {cv_scores.mean():.4f} (95% CI [{lo:.4f}, {hi:.4f}])')
print(f'  CV F1:       {cv_f1.mean():.4f}')
print(f'\n[SHAP Top 3]')
for i, (f, v) in enumerate(top.head(3).items(), 1):
    print(f'  {i}. {f:15s} {v:.4f}')
print(f'\n[Robustness] subset별 Top1 feature')
for _, r in rb_df.iterrows():
    print(f'  {r["subset"]:20s} (N={r["N"]}) → Top1={r["Top1"]}')
consistent = rb_df['Top1'].nunique() == 1
print(f'\n  {"✅ 모든 subset에서 일관 (EA가 Top)" if consistent else "⚠️ subset별 불일치"}')
print('\n[Interaction] EA × EQ 모두 높을 때만 Platinum 근접')
print('[결론] 에너지(EA) 카테고리가 한국 LEED 등급의 최대 결정 요인')

  LEEDGRAPH 최종 분석 결과 요약

[데이터] N=460 (Gold 51% / Silver 26% / Platinum 12% / Certified 11%)

[모델 비교 CV Accuracy]
  RandomForest    0.8457 ± 0.0081
  LogReg          0.8391 ± 0.0379
  LightGBM        0.8370 ± 0.0322
  XGBoost         0.8239 ± 0.0303

[최종 RandomForest]
  CV Accuracy: 0.8457 (95% CI [0.8391, 0.8522])
  CV F1:       0.8401

[SHAP Top 3]
  1. ratio_EA        0.0855
  2. ratio_EQ        0.0668
  3. ratio_WE        0.0492

[Robustness] subset별 Top1 feature
  전체                   (N=460) → Top1=ratio_EA
  credit_hit>0.7       (N=324) → Top1=ratio_EA
  구버전(v2.x, v2009)     (N=136) → Top1=ratio_EA
  신버전(v4, v4.1)        (N=324) → Top1=ratio_EA
  Gold 이상              (N=291) → Top1=ratio_EA
  연면적 > 10000sqm       (N=284) → Top1=ratio_EA
  LLM 재매핑(drift>20%)   (N=32) → Top1=log_area

  ⚠️ subset별 불일치

[Interaction] EA × EQ 모두 높을 때만 Platinum 근접
[결론] 에너지(EA) 카테고리가 한국 LEED 등급의 최대 결정 요인


## 생성된 산출물

### Tables (`results/tables/`)
- Table2_performance.csv — RandomForest 성능 + CI
- Table3_shap_top.csv — SHAP feature 순위
- Table4_robustness.csv — subset별 성능
- Table5_shap_by_grade.csv — 등급별 SHAP
- Table_model_comparison.csv — 4개 모델 비교
- Table_fold_stability.csv — CV fold별 feature importance

### Figures (`results/figures/`)
- Figure1_shap_importance.png — SHAP bar
- Figure2_shap_beeswarm.png — beeswarm plot
- robust_shap_llm_remapped.png — LLM 재매핑 subset SHAP
- eda_grade_category_heatmap.png — 등급×카테고리
- eda_correlation_heatmap.png — 카테고리 상관관계
- eda_version_score_boxplot.png — 버전별 v5 점수
- eda_drift_distribution.png — drift 분포
- confusion_matrix_train.png — train 혼동행렬
- confusion_matrix_cv.png — CV 혼동행렬
- shap_class_heatmap.png — 등급별 SHAP
- shap_dependence.png — EA/EQ 의존성
- interaction_EA_EQ.png — EA×EQ 등급 매트릭스

## Section 3.4 — 표준화 결과 검증 표 산출

In [27]:
# Section 3.4 검증 표는 독립 재현 스크립트로 분리되었다:
#   scripts/generate_table_3_4_consistency.py  (PDF 재파싱 기반 독립 검증)
#   scripts/generate_table_3_4_drift.py
# 기존 셀의 카테고리합 vs total_score_v5 비교는 정의상 항상 일치하는
# 동어반복 검증이라 폐기하고, 위 스크립트로 대체한다.
print('Section 3.4 표는 scripts/generate_table_3_4_*.py 로 생성합니다.')

Section 3.4 표는 scripts/generate_table_3_4_*.py 로 생성합니다.
